In [1]:
import PeterChurchillFunctions as Function
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf
import statsmodels.api as sm
from scipy.odr import ODR, Model, RealData
from matplotlib.colors import LogNorm
from dask.distributed import Client
client = Client()

In [2]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 8
Total threads: 32,Total memory: 125.70 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44687,Workers: 8
Dashboard: http://127.0.0.1:8787/status,Total threads: 32
Started: Just now,Total memory: 125.70 GiB
Comm: tcp://127.0.0.1:36701,Total threads: 4
Dashboard: http://127.0.0.1:35837/status,Memory: 15.71 GiB
Nanny: tcp://127.0.0.1:39275,


In [3]:
def Suceptibility_by_Level(CCN_ds, CDNC_da):
    """
    Compute OLS slope/intercept between CCN(radius, lev, time)
    and CDNC(lev, time) across 'time'.
    """
    CCN_aligned, CDNC_aligned = xr.align(CCN_ds, CDNC_da)

    OLS_slope, OLS_intercept = xr.apply_ufunc(
        Function.OLS_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_aligned),
        input_core_dims=[['time'], ['time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )
    ODR_slope, ODR_intercept = xr.apply_ufunc(
        Function.TLS_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_aligned),
        input_core_dims=[['time'], ['time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )
    Deming_slope, Deming_intercept = xr.apply_ufunc(
        Function.deming_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_aligned),
        input_core_dims=[['time'], ['time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )
    PCA_slope, PCA_intercept = xr.apply_ufunc(
        Function.PCA_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_aligned),
        input_core_dims=[['time'], ['time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )

    ds_out = xr.Dataset(
        data_vars={
            'OLS slope': (('radius', 'lev'), OLS_slope.data),
            'OLS intercept': (('radius', 'lev'), OLS_intercept.data),
            'ODR slope': (('radius', 'lev'), ODR_slope.data),
            'ODR intercept': (('radius', 'lev'), ODR_intercept.data),
            'Deming slope': (('radius', 'lev'), Deming_slope.data),
            'Deming intercept': (('radius', 'lev'), Deming_intercept.data),
            'PCA slope': (('radius', 'lev'), PCA_slope.data),
            'PCA intercept': (('radius', 'lev'), PCA_intercept.data),

        },
        coords={
            'radius': CCN_aligned.radius,
            'lev': CCN_aligned.lev,

        }
    )
    ds_out2 = xr.Dataset(
        data_vars={
            'CCN': CCN_aligned,       # dims: (radius, lev, time)
            'CDNC': CDNC_aligned      # dims: (lev, time)
        },
        coords={
            'radius': CCN_aligned.radius,
            'lev': CCN_aligned.lev,
            'time': CCN_aligned.time
        }
    )

    return ds_out, ds_out2


In [4]:
def compute_allLev(CCN_ds, CDNC_da):
    """
    Compute OLS and TLS slope/intercept per radius,
    flattening across all levels and times.
    Returns a Dataset with variables: slope_OLS, intercept_OLS, slope_TLS, intercept_TLS
    """
    # Align CCN and CDNC over lev and time
    CCN_aligned, CDNC_aligned = xr.align(CCN_ds, CDNC_da)

    # Broadcast CDNC to match CCN dimensions
    CDNC_broadcast = CDNC_aligned.broadcast_like(CCN_aligned)

    # --- OLS ---
    slope_OLS, intercept_OLS = xr.apply_ufunc(
        Function.OLS_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_broadcast),
        input_core_dims=[['lev', 'time'], ['lev', 'time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )

    # --- TLS ---
    slope_TLS, intercept_TLS = xr.apply_ufunc(
        Function.TLS_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_broadcast),
        input_core_dims=[['lev', 'time'], ['lev', 'time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )
    
    # --- Deming ---
    slope_Deming, intercept_Deming = xr.apply_ufunc(
        Function.deming_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_broadcast),
        input_core_dims=[['lev', 'time'], ['lev', 'time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )
    # --- PCA ---
    slope_PCA, intercept_PCA = xr.apply_ufunc(
        Function.PCA_fit,
        np.log10(CCN_aligned),
        np.log10(CDNC_broadcast),
        input_core_dims=[['lev', 'time'], ['lev', 'time']],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )


    # Package into dataset
    ds_out = xr.Dataset(
        data_vars={
            'All_Level_OLS_slope': (('radius',), slope_OLS.data),
            'All_Level_OLS_intercept': (('radius',), intercept_OLS.data),
            'All_Level_TLS_slope': (('radius',), slope_TLS.data),
            'All_Level_TLS_intercept': (('radius',), intercept_TLS.data),
            'All_Level_Deming_slope': (('radius',), slope_Deming.data),
            'All_Level_Deming_intercept': (('radius',), intercept_Deming.data),
            'All_Level_PCA_slope': (('radius',), slope_PCA.data),
            'All_Level_PCA_intercept': (('radius',), intercept_PCA.data),

        },
        coords={
            'radius': CCN_aligned.radius,

        }
    )

    return ds_out

In [5]:
ECPath = "/share/sabl0586/all_stations_EC-Earth_PRCP2SZDST_ilevall_levs_4Peter.nc"
ds = xr.open_dataset(ECPath, chunks={})
stations = ds["station"].values
TestStation = ['SMR-II']
radii = np.arange(20,51)
VarList = []
IFSVarList = []
x = xr.DataArray(np.logspace(-0.5,3, num=200), dims =['D'], coords= {'D':np.logspace(-0.5,3, num=200)})

In [ ]:
ds_ifs=xr.Dataset()

SMRECEarth['lev']=SMRECEarth['pressure'].mean('time')
for ifs in ifs_vars:
    ds_ifs[ifs] = SMRECEarth[ifs]
ds_ifs = ds_ifs[['var130','var54','var131','var132','var132','var20','var21','var22','var248']].isel(lev=0).drop_vars('lev')
ds_ifs['lev_ifs'] = ds_ifs['var54'].mean('time')
for ifs in ifs_vars:
    ds_ifs[ifs] = ds_ifs.sel(lev_ifs = SMRECEarth['lev'], method='nearest')[ifs]

    
ds_ifs = ds_ifs.drop_vars('lev_ifs')
ds_ifs['lev'] = ds_ifs['lev']/100




In [6]:
def ECearthExtract_Dask(ECPath, station, chunks="auto"):
    """
    Lazily load EC-Earth data using Dask and optionally compute PNSD.

    Parameters
    ----------
    ECPath : str
        Path to EC-Earth NetCDF file.
    station : str
        Station name or coordinate.
    chunks : str or dict, optional
        Dask chunking specification. Default 'auto'.
    """

    import xarray as xr

    #Open lazily with Dask
    Data = xr.open_dataset(ECPath, chunks=chunks)
    Data = Data.sel(station=station)

    ds = xr.Dataset()
    PNSD_ds = xr.Dataset()
    cdnc = xr.Dataset()
    ds_IFS = xr.Dataset()


    #Define variables for the ERF
    radius_variables = ['RDRY_NUS', 'RDRY_AIS', 'RDRY_ACS', 'RWET_AII', 'RDRY_COS', 'RWET_ACI', 'RWET_COI']
    Numb_variables = ['N_NUS', 'N_AIS', 'N_ACS', 'N_AII', 'N_COS', 'N_ACI', 'N_COI']
    ModesSigma = [1.59, 1.59, 1.59, 2.0, 1.59, 1.59, 2.0]
    
    for radius, number in zip(radius_variables, Numb_variables):
        if radius in Data and number in Data:
            ds[radius] = Data[radius]
            print(f' {radius} added to Dataset')
            ds[number] = Data[radius]
            print(f' {number} added to Dataset')
        else:
            print(f'{radius, number} not found in EC Path Data')
            
    # ADD PRESSURE TO DATASET        
    ds['pressure'] = Data['pressure']
    
        #Clean up radius variables (convert to nm if needed)
    for r in radius_variables:
        if r in Data:
            ds[r] = Data[r].where(Data[r] > 0)
            if "units" in Data[r].attrs and Data[r].attrs["units"] == "m":
                ds[r] = ds[r] * 1e9
                ds[r].attrs["units"] = "nm"
                print(f'Converting {r} to nm')

    #Converting the IFS levels to match the EC-Earth Levs
    for ifs in [['var20', 'var22', 'var54']]:
        ds_IFS[ifs] = Data[ifs]
    ds_IFS = ds_IFS[['var20', 'var22', 'var54']].isel(lev=0).drop_vars('lev')
    ds_IFS['lev_ifs'] = ds_IFS['var54'].mean('time')
    """
 
    for ifs in ['var20', 'var22']:
        ds_IFS[ifs] = ds_IFS.sel(lev_ifs = Data['lev'], method='nearest')[ifs]
    ds_IFS = ds_IFS.drop_vars('lev_ifs')
    ds_IFS['lev'] = ds_IFS['lev']/100

            #Pressure to hPa
    ds["lev"] = Data["pressure"].mean("time") / 100
    ds["lev"].attrs["units"] = "hPa" 
    print('Converting lev to hPa')
    
    # Compute CDNC and drop lev_ifs
    cdnc = (ds_IFS["var20"] / dsIFS["var22"]).where(ds_IFS["var22"] > 0).compute()
    """

    
    return ds, ds_IFS
    


In [7]:
EC_ds, ds_ifs = ECearthExtract_Dask(ECPath, TestStation, chunks="auto")
        #Pressure to ds

EC_ds['lev'] = EC_ds['pressure'].mean('time')
EC_ds['lev'] = EC_ds.lev.compute()
ds_ifs['lev_ifs'] = ds_ifs.lev_ifs.compute()

 RDRY_NUS added to Dataset
 N_NUS added to Dataset
 RDRY_AIS added to Dataset
 N_AIS added to Dataset
 RDRY_ACS added to Dataset
 N_ACS added to Dataset
 RWET_AII added to Dataset
 N_AII added to Dataset
 RDRY_COS added to Dataset
 N_COS added to Dataset
 RWET_ACI added to Dataset
 N_ACI added to Dataset
 RWET_COI added to Dataset
 N_COI added to Dataset
Converting RDRY_NUS to nm
Converting RDRY_AIS to nm
Converting RDRY_ACS to nm
Converting RWET_AII to nm
Converting RDRY_COS to nm
Converting RWET_ACI to nm
Converting RWET_COI to nm


In [8]:
print(EC_ds)
print(ds_ifs)

<xarray.Dataset> Size: 126MB
Dimensions:    (station: 1, lev: 34, time: 61369)
Coordinates:
    lev        (station, lev) float32 136B 9.989e+04 9.873e+04 ... 48.77 10.71
  * station    (station) <U8 32B 'SMR-II'
  * time       (time) datetime64[ns] 491kB 2012-01-01 ... 2019-01-01
    time_orig  (time) datetime64[ns] 491kB dask.array<chunksize=(61369,), meta=np.ndarray>
Data variables: (12/15)
    RDRY_NUS   (station, time, lev) float32 8MB dask.array<chunksize=(1, 61369, 34), meta=np.ndarray>
    N_NUS      (station, time, lev) float32 8MB dask.array<chunksize=(1, 61369, 34), meta=np.ndarray>
    RDRY_AIS   (station, time, lev) float32 8MB dask.array<chunksize=(1, 61369, 34), meta=np.ndarray>
    N_AIS      (station, time, lev) float32 8MB dask.array<chunksize=(1, 61369, 34), meta=np.ndarray>
    RDRY_ACS   (station, time, lev) float32 8MB dask.array<chunksize=(1, 61369, 34), meta=np.ndarray>
    N_ACS      (station, time, lev) float32 8MB dask.array<chunksize=(1, 61369, 34), meta=np.

In [19]:

lev_target = EC_ds.sel(station=TestStation)['lev'].isel(station=0)
ds_ifs['var20'] = ds_ifs['var20'].interp(lev_ifs=lev_target, method='nearest')
#for ifs in ['var20', 'var22']:
  #  ds_ifs[ifs] = ds_ifs[ifs].interp(lev_ifs=lev_target, method='nearest')

ValueError: Input DataArray is not 1-D.

In [13]:
EC_ds['lev'].

<xarray.DataArray 'lev' (station: 1, lev: 34)> Size: 136B
array([[9.9892391e+04, 9.8732625e+04, 9.6420477e+04, 9.2849844e+04,
        8.7912352e+04, 8.1704844e+04, 7.4441578e+04, 6.6418125e+04,
        5.7961582e+04, 4.9510828e+04, 4.1656543e+04, 3.4813586e+04,
        2.9839766e+04, 2.6376936e+04, 2.3281311e+04, 2.0515561e+04,
        1.8051818e+04, 1.5863072e+04, 1.3902802e+04, 1.2173625e+04,
        1.0627670e+04, 9.2398945e+03, 7.9595459e+03, 6.7739370e+03,
        5.4300879e+03, 4.0125781e+03, 2.8259700e+03, 1.8781792e+03,
        1.1601819e+03, 6.5437256e+02, 3.2782886e+02, 1.4046919e+02,
        4.8767887e+01, 1.0707044e+01]], dtype=float32)
Coordinates:
    lev      (station, lev) float32 136B 9.989e+04 9.873e+04 ... 48.77 10.71
  * station  (station) <U8 32B 'SMR-II'

In [48]:
ds_ifs
for ifs in ['var20', 'var22']:
    ds_IFS[ifs] = ds_IFS.[ifs].interp(lev_ifs=lev_target, method='nearest')
ds_IFS = ds_IFS.drop_vars('lev_ifs')
ds_IFS['lev'] = ds_IFS['lev']/100



# Compute CDNC and drop lev_ifs
cdnc = (ds_IFS["var20"] / dsIFS["var22"]).where(ds_IFS["var22"] > 0).compute()

<xarray.Dataset> Size: 68MB
Dimensions:    (station: 1, time: 61369, lev_ifs: 91)
Coordinates:
  * station    (station) <U8 32B 'SMR-II'
  * time       (time) datetime64[ns] 491kB 2012-01-01 ... 2019-01-01
    time_orig  (time) datetime64[ns] 491kB dask.array<chunksize=(61369,), meta=np.ndarray>
    lev_ifs    (station, lev_ifs) float32 364B dask.array<chunksize=(1, 31), meta=np.ndarray>
Data variables:
    var20      (station, time, lev_ifs) float32 22MB dask.array<chunksize=(1, 24597, 31), meta=np.ndarray>
    var22      (station, time, lev_ifs) float32 22MB dask.array<chunksize=(1, 24597, 31), meta=np.ndarray>
    var54      (station, time, lev_ifs) float32 22MB dask.array<chunksize=(1, 24597, 31), meta=np.ndarray>

In [45]:
CDNC_ds.isel(lev_ifs = 1).dropna('time').mean('time').plot(marker = 'x')

KeyboardInterrupt: 

In [8]:
for lev in CDNC_ds.lev:
    print(CDNC_ds.sel(lev = lev).mean().values)

nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan


In [ ]:
all_stations = []
all_stationsCCN = []

for station in stations:
    Nor_ds = Function.NorESMExtract_Dask(NorPath, station, VarList, x, PNSD=False)
    CCN_ds = Function.NorERF(Nor_ds, radii)
    
    Level_susc_ds, Levels_CCN_ds = Suceptibility_by_Level(CCN_ds, Nor_ds['CDNC'])
    Susceptibility_AllLevs = compute_allLev(CCN_ds, Nor_ds['CDNC'])
    
    Susceptibility_ds = xr.merge([Level_susc_ds, Susceptibility_AllLevs])
    
    all_stations.append(Susceptibility_ds.assign_coords(station=station))
    all_stationsCCN.append(Levels_CCN_ds.assign_coords(station=station))

# Concatenate across stations
Susceptibility_all = xr.concat(all_stations, dim='station')
CCN_all = xr.concat(all_stationsCCN, dim='station')

In [ ]:
#Susceptibility_all = Susceptibility_all.compute()
#CCN_all = CCN_all.compute()

In [ ]:
def PLOTSusc_by_Level(ds = Susceptibility_all, Fit = 'OLS', station = 'SMR-II'):
    radii_to_plot = [20, 30, 40, 50]
    for r in radii_to_plot:
        ds[f'{Fit} slope'].sel(radius=r, station = station).plot(
            y='lev', 
            yincrease=False, 
            ylim=[1000, 600],
            xlim = [0,1],
            label=f'radius={r} nm'
        )
    
    plt.legend()
    plt.xlabel("Susceptibility")
    plt.ylabel("Pressure Level [hPa]")
    plt.title(f"Susceptibility ({Fit}) with height for selected cutoff radii")
    plt.show()

FitType = ['OLS', 'ODR', 'Deming', 'PCA']
for fit in FitType:
    print(fit)
    PLOTSusc_by_Level(Susceptibility_all, Fit = fit, station = 'ATTO')

In [ ]:
stations = Susceptibility_all.station.values
methods = ["OLS", "TLS", "Deming", "PCA"]

# Define number of columns and rows dynamically
ncols = 4
nrows = int(np.ceil(len(stations) / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(5 * ncols, 4 * nrows),
    sharey=True
)

axes = axes.flatten()

for i, station in enumerate(stations):
    ax = axes[i]
    for method in methods:
        var_name = f"All_Level_{method}_slope"
        if var_name in Susceptibility_all:
            Susceptibility_all[var_name].sel(station=station).plot(
                ax=ax, marker='x', label=method
            )
    ax.set_title(f"{station}", fontsize=11)
    ax.set_xlabel("Cutoff Radius (nm)")
    ax.set_ylabel("Susceptibility (slope)")
    ax.set_ylim(0, 2)
    ax.grid(True, alpha=0.3)

# Hide any unused subplots
for j in range(len(stations), len(axes)):
    axes[j].set_visible(False)

# Add a global legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Method", loc="lower right", fontsize=20)

fig.suptitle("All-level CCN–CDNC Susceptibility per Station", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
def plot_station_fits_from_dataset(Susceptibility_all, station, CCN_ds, EC_ds, radius=35, methods=None):
    """
    Plot Nd vs CCN (log–log) with precomputed fits from Susceptibility_all.
    No refitting is done here.
    """
    if methods is None:
        methods = ["OLS", "TLS", "Deming", "PCA"]

    # --- Prepare data ---
    x = CCN_all['CCN'].sel(radius=radius, station = station).to_numpy().ravel()
    y = CCN_all["CDNC"].sel(station = station).to_numpy().ravel()

    mask = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x, y = x[mask], y[mask]

    fits = []

    # --- Get slopes/intercepts from dataset ---
    for method in methods:
        slope_var = f"All_Level_{method}_slope"
        intercept_var = f"All_Level_{method}_intercept"

        if slope_var in Susceptibility_all and intercept_var in Susceptibility_all:
            slope = Susceptibility_all[slope_var].sel(station=station, radius=radius).compute().item()
            intercept = Susceptibility_all[intercept_var].sel(station=station, radius=radius).compute().item()

            fits.append({
                "slope": slope,
                "intercept": intercept,
                "label": f"{method}: slope={slope:.2f}",
                "style": {
                    "OLS": "r-",
                    "TLS": "g--",
                    "Deming": "b-.",
                    "PCA": "m:",
                }.get(method, "k-")
            })

    # --- Plot all fits on a single hexbin plot ---
    fig, ax = Function.plot_hexbin_regression_multi(
        x, y,
        fits=fits,
        lims=(1, 1e4),
        title=f"{station}: CCN–CDNC Fits (radius={radius} nm)"
    )

    plt.tight_layout()
    return fig, ax

In [ ]:
for station in stations:
    plot_station_fits_from_dataset(
        Susceptibility_all,
        station=station,
        CCN_ds=CCN_ds,
        EC_ds=EC_ds,
        radius=35
    )

In [ ]:
CCN_all

In [ ]:
def PNSD_ECearth(PNSD, Data)       
    #Particle Number Size Distribution
    if PNSD:
        # Collect PNSD variables
        for radius, conc in zip(radius_variables, Numb_variables):
            if radius in Data and conc in Data:
                PNSD_ds[radius] = Data[radius]
                PNSD_ds[conc] = Data[conc]

        # Compute distributions lazily with Dask
        dis_variable = ["NUS_dis", "AIS_dis", "ACS_dis", "COS_dis", "AII_dis", "ACI_dis", "COI_dis"]

        for radius, conc, sigma, dist in zip(radius_variables, Numb_variables, ModesSigma, dis_variable):
            if radius not in PNSD_ds or conc not in PNSD_ds:
                continue

            PNSD_ds[dist] = xr.apply_ufunc(
                Function.dNdlogD,
                PNSD_ds[conc],
                Xspace,
                PNSD_ds[radius] * 2,
                sigma,
                dask="parallelized",
                output_dtypes=[float],
            )

        # Combine all dNdlogD modes lazily
        dNdlogD_vars = [v for v in PNSD_ds.data_vars if v.endswith("_dis")]
        if len(dNdlogD_vars) > 0:
            PNSD_ds["dNdlogD"] = sum(PNSD_ds[v] for v in dNdlogD_vars)

        return ds, PNSD_ds, cdnc

    else:
        # Return only basic variables
        for radius, conc in zip(radius_variables, Numb_variables):
            if radius in Data:
                ds[radius] = Data[radius]
            if conc in Data:
                ds[conc] = Data[conc]
                
        return ds, cdnc